# Notebook 02 — Feature Engineering (Stage 1, continued)
Turns the clean historical table from Notebook 01 into ML-ready features: temporal, lag, rolling-average, and spatial features.

**Input:** `tonnage_clean.parquet`, `districts.geojson`
**Output:** `features.parquet`

In [1]:
# Setup and load Stage 1 output, sorted by district then date
# (sorting is required before computing lag/rolling features per district).
import pandas as pd
import geopandas as gpd
from pathlib import Path

PROJECT_ROOT = Path(r"C:\Users\pande\OneDrive\Documents\Waste management grid")
PROCESSED = PROJECT_ROOT / "data" / "processed"

tonnage = pd.read_parquet(PROCESSED / "tonnage_clean.parquet")
tonnage = tonnage.sort_values(["BoroCd", "date"]).reset_index(drop=True)
print(tonnage.shape)
tonnage.head()

(25119, 15)


,MONTH,BOROUGH,COMMUNITYDISTRICT,REFUSETONSCOLLECTED,PAPERTONSCOLLECTED,MGPTONSCOLLECTED,RESORGANICSTONS,SCHOOLORGANICTONS,LEAVESORGANICTONS,XMASTREETONS,OTHERORGANICSTONS,BOROUGH_ID,date,BoroCd,total_tons
0,1990 / 10,Manhattan,1,374.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,1990-10-01,101,374.0
1,1990 / 11,Manhattan,1,704.3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,1990-11-01,101,704.3
2,1990 / 12,Manhattan,1,737.6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,1990-12-01,101,737.6
3,1991 / 01,Manhattan,1,771.3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,1991-01-01,101,771.3
4,1991 / 02,Manhattan,1,672.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,1991-02-01,101,672.1


In [2]:
# Temporal features: calendar signals the model can use to learn seasonality.
tonnage["year"] = tonnage["date"].dt.year
tonnage["month_num"] = tonnage["date"].dt.month
tonnage["quarter"] = tonnage["date"].dt.quarter
tonnage["is_summer"] = tonnage["month_num"].isin([6, 7, 8]).astype(int)
tonnage["is_winter"] = tonnage["month_num"].isin([12, 1, 2]).astype(int)

tonnage[["date", "year", "month_num", "quarter", "is_summer", "is_winter"]].head()

,date,year,month_num,quarter,is_summer,is_winter
0,1990-10-01,1990,10,4,0,0
1,1990-11-01,1990,11,4,0,0
2,1990-12-01,1990,12,4,0,1
3,1991-01-01,1991,1,1,0,1
4,1991-02-01,1991,2,1,0,1


In [3]:
# Lag features (per district): last month's and same-month-last-year's waste,
# among the strongest predictors for next month's waste generation.
tonnage["total_tons_lag1"] = tonnage.groupby("BoroCd")["total_tons"].shift(1)
tonnage["total_tons_lag12"] = tonnage.groupby("BoroCd")["total_tons"].shift(12)

tonnage[tonnage["BoroCd"] == 201][["date", "total_tons", "total_tons_lag1", "total_tons_lag12"]].head(15)

,date,total_tons,total_tons_lag1,total_tons_lag12
5161,1991-04-01,18.8,NaN,NaN
5162,1991-05-01,2596.6,18.8,NaN
5163,1991-06-01,2492.9,2596.6,NaN
5164,1991-07-01,2626.9,2492.9,NaN
5165,1991-08-01,2538.7,2626.9,NaN
5166,1991-09-01,2485.1,2538.7,NaN
5167,1991-10-01,2628.0,2485.1,NaN
5168,1991-11-01,2510.7,2628.0,NaN
5169,1991-12-01,2723.6,2510.7,NaN
5170,1992-01-01,2462.2,2723.6,NaN


In [4]:
# Rolling averages (per district): 3-month and 12-month trailing averages, smoothing out noise.
# shift(1) BEFORE rolling is deliberate — it excludes the current month from its own average,
# avoiding data leakage.
tonnage["rolling_3m_avg"] = (
    tonnage.groupby("BoroCd")["total_tons"]
    .transform(lambda x: x.shift(1).rolling(3).mean())
)
tonnage["rolling_12m_avg"] = (
    tonnage.groupby("BoroCd")["total_tons"]
    .transform(lambda x: x.shift(1).rolling(12).mean())
)

tonnage[tonnage["BoroCd"] == 201][["date", "total_tons", "rolling_3m_avg", "rolling_12m_avg"]].head(15)

,date,total_tons,rolling_3m_avg,rolling_12m_avg
5161,1991-04-01,18.8,NaN,NaN
5162,1991-05-01,2596.6,NaN,NaN
5163,1991-06-01,2492.9,NaN,NaN
5164,1991-07-01,2626.9,1702.766667,NaN
5165,1991-08-01,2538.7,2572.133333,NaN
5166,1991-09-01,2485.1,2552.833333,NaN
5167,1991-10-01,2628.0,2550.233333,NaN
5168,1991-11-01,2510.7,2550.600000,NaN
5169,1991-12-01,2723.6,2541.266667,NaN
5170,1992-01-01,2462.2,2620.766667,NaN


In [5]:
# Spatial features: district area and centroid coordinates, derived from the district boundary
# shapes. Projected to EPSG:2263 (feet) for accurate area/centroid math, then converted back
# to EPSG:4326 (lat/lon) for use as model features and for mapping later.
districts = gpd.read_file(PROCESSED / "districts.geojson")
districts_proj = districts.to_crs(epsg=2263)

districts["area_sqm"] = districts_proj.geometry.area
centroids = districts_proj.geometry.centroid.to_crs(epsg=4326)
districts["centroid_lat"] = centroids.y
districts["centroid_lon"] = centroids.x

districts[["BoroCd", "area_sqm", "centroid_lat", "centroid_lon"]].head()

C:\Users\pande\anaconda3\Lib\site-packages\pyogrio\core.py:35: RuntimeWarning: Could not detect GDAL data files. Set GDAL_DATA environment variable to the correct path.
  _init_gdal_data()


,BoroCd,area_sqm,centroid_lat,centroid_lon
0,410,1.720777e+08,40.669367,-73.832737
1,483,1.919977e+08,40.646984,-73.786109
2,226,5.056510e+07,40.897060,-73.886611
3,313,8.826481e+07,40.580648,-73.982151
4,105,4.379007e+07,40.752276,-73.984057


In [6]:
# Merge spatial features into the main table, then drop rows that can't have a full
# lag/rolling history (each district's first ~12 months, before lag12 exists).
spatial_cols = districts[["BoroCd", "area_sqm", "centroid_lat", "centroid_lon"]]
features = tonnage.merge(spatial_cols, on="BoroCd", how="left")
print("Unmatched BoroCd after spatial merge:", features["area_sqm"].isna().sum())

print("Rows before dropping lag-NaN:", len(features))
features_clean = features.dropna(subset=["total_tons_lag1", "total_tons_lag12"])
print("Rows after:", len(features_clean))
features_clean.head()

Unmatched BoroCd after spatial merge: 0
Rows before dropping lag-NaN: 25119
Rows after: 24411


,MONTH,BOROUGH,COMMUNITYDISTRICT,REFUSETONSCOLLECTED,PAPERTONSCOLLECTED,MGPTONSCOLLECTED,RESORGANICSTONS,SCHOOLORGANICTONS,LEAVESORGANICTONS,XMASTREETONS,...,quarter,is_summer,is_winter,total_tons_lag1,total_tons_lag12,rolling_3m_avg,rolling_12m_avg,area_sqm,centroid_lat,centroid_lon
12,1991 / 10,Manhattan,1,759.8,0.0,0.0,0.0,0.0,0.0,0.0,...,4,0,0,686.4,374.0,703.466667,699.466667,4.169702e+07,40.70738,-74.011997
13,1991 / 11,Manhattan,1,701.3,0.0,0.0,0.0,0.0,0.0,0.0,...,4,0,0,759.8,704.3,713.766667,731.616667,4.169702e+07,40.70738,-74.011997
14,1991 / 12,Manhattan,1,712.9,0.0,0.0,0.0,0.0,0.0,0.0,...,4,0,1,701.3,737.6,715.833333,731.366667,4.169702e+07,40.70738,-74.011997
15,1992 / 01,Manhattan,1,725.5,0.0,0.0,0.0,0.0,0.0,0.0,...,1,0,1,712.9,771.3,724.666667,729.308333,4.169702e+07,40.70738,-74.011997
16,1992 / 02,Manhattan,1,655.5,0.0,0.0,0.0,0.0,0.0,0.0,...,1,0,1,725.5,672.1,713.233333,725.491667,4.169702e+07,40.70738,-74.011997


In [7]:
# Save Stage 1 feature-engineered output, then reload and verify integrity
# (no leftover NaNs in lag/rolling columns, expected district count).
features_clean.to_parquet(PROCESSED / "features.parquet", index=False)

check = pd.read_parquet(PROCESSED / "features.parquet")
print("Saved:", PROCESSED / "features.parquet")
print("\nNaN check (should all be 0):")
print(check[["total_tons_lag1", "total_tons_lag12", "rolling_3m_avg", "rolling_12m_avg"]].isna().sum())
print("\nTotal districts:", check["BoroCd"].nunique())
print("Date range:", check["date"].min(), "to", check["date"].max())
print("\nNotebook 02 complete.", check.shape[0], "rows.")

Saved: C:\Users\pande\OneDrive\Documents\Waste management grid\data\processed\features.parquet

NaN check (should all be 0):
total_tons_lag1     0
total_tons_lag12    0
rolling_3m_avg      0
rolling_12m_avg     0
dtype: int64

Total districts: 59
Date range: 1991-05-01 00:00:00 to 2026-06-01 00:00:00

Notebook 02 complete. 24411 rows.
